# Closure-Certified Text → Logic Deduction — Operational Case Study

**text → span-local atomic read → QCN closure → runnable SWI-Prolog trace → quantified hallucination reduction**

This notebook is a **runnable, LLM-free** distillation of the experiment
*"Operational ~3000-char end-to-end closure-certified deduction case study"*.

The contribution is a **closure certificate with an abstain-on-collapse output contract**: a
neuro-symbolic pipeline reads atomic relations span-locally with an LLM, then a **qualitative
constraint-network (QCN) closure engine** either

* **emits** a relation when path-consistency narrows the held-out query edge to a **singleton**, or
* **abstains** when the local reads under-determine the order (non-singleton), or
* flags a **Mode-B** inconsistency certificate (empty intersection).

Because the closure *abstains* exactly where the evidence is insufficient, it is
**hallucination-safe by construction** — the headline metric is the **confident-wrong (hallucination)
reduction** versus a raw LLM, reported as a risk–coverage tradeoff.

**What runs here (no API key, no cost):**
1. The **POINT-algebra closure engine** (`engine.py`, verbatim) reproduces the published
   **emit / abstain** decisions on the *real* LLM span-local reads stored in the data.
2. The **hallucination-reduction** headline is recomputed from the cached per-query predictions.
3. The **kinship forward-closure least-fixpoint engine** (`kinship.py`, verbatim) runs worked
   multi-hop chains + an absent-relation pair, then discharges a **runnable SWI-Prolog** program
   (`prolog_kinship.py`, verbatim) — executed in `swipl` when available, else the authoritative
   Python engine.

The LLM extraction step (text → atomic edges) is **measured, not reproduced** here; its cached
outputs are loaded from `mini_demo_data.json`.

In [ ]:
# --- Dependencies -------------------------------------------------------------
# The closure engines are PURE PYTHON (stdlib only). We only need numpy + matplotlib
# for the summary plots, plus (optionally) SWI-Prolog to EXECUTE the discharged programs.
import subprocess, sys, shutil

def _pip(*a):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# numpy + matplotlib are pre-installed on Colab -> install locally only to match Colab.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

# SWI-Prolog is OPTIONAL: the kinship discharge EXECUTES in swipl when present; otherwise
# the Python forward-closure engine is authoritative (the artifact's honest fallback).
if shutil.which('swipl') is None:
    try:
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'swi-prolog'],
                       check=False, timeout=180,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        pass
print('SWI-Prolog available:', shutil.which('swipl') is not None)

In [ ]:
# --- Imports ------------------------------------------------------------------
import json, os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(0)

In [ ]:
# --- Data loading (GitHub URL with local fallback, Colab-compatible) -----------
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-40a89b-no-derivation-no-relation-a-closure-cert/main/round-5/experiment-3/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("dataset:", data["dataset"])
print("temporal example rows:", len(data["examples"]),
      "| docs:", sorted(set(e["metadata_docid"] for e in data["examples"])))
print("temporal trace-graphs:", len(data["temporal_trace_graphs"]))
print("kinship worked chains:", [k["name"] for k in data["kinship_worked_examples"]])
print("kinship relation types:", data["kinship_composition_table"]["relation_types"].__len__(),
      "->", list(data["kinship_composition_table"]["relation_types"].keys()))

## Configuration

Everything here is instant pure-Python closure, so these knobs only bound *how much* of the
cached data we replay. They start small for a quick smoke test; the full demo simply replays
all rows (still well under a second of compute).

In [ ]:
# --- Config (tunable) ---------------------------------------------------------
# Start values are the smallest that still produce output; the production demo uses ALL data.
MAX_TEMPORAL_EXAMPLES = len(data["examples"])   # per-query rows scored for hallucination reduction
                                                # (minimum smoke value: 30 = one document)
MAX_TRACE_GRAPHS      = len(data["temporal_trace_graphs"])  # closure decisions to reproduce live (min: 2)
MAX_KINSHIP_EXAMPLES  = len(data["kinship_worked_examples"])  # worked kinship chains to run (min: 1)
RUN_SWIPL             = True   # execute discharged Prolog in swipl if installed, else Python fallback

print(f"replay: {MAX_TEMPORAL_EXAMPLES} temporal rows, {MAX_TRACE_GRAPHS} trace-graphs, "
      f"{MAX_KINSHIP_EXAMPLES} kinship chains")

## 1 · The QCN closure engine (`engine.py`, verbatim)

The symbolic core. The **convex point algebra over event start-points** `{<, =, >}` is
**path-consistency complete** (an exact inconsistency certificate). `pc2_full` is Mackworth's
PC-2 worklist closure to fixpoint (**our method**); `naive_single_pass` is the one-pass
baseline (the *iteration* contrast). This is the artifact's `engine.py`, unchanged.

In [ ]:
#!/usr/bin/env python3
"""Self-contained qualitative-constraint-network (QCN) closure engine.

Two algebras built PROGRAMMATICALLY (no external table files), each cross-checked
against the verified GQR composition cells from the implementation dossier:

  * POINT  -- convex point algebra over event START-points {<,=,>} (PC COMPLETE;
              EXACT inconsistency certificate). The ONLY non-convex relation is
              `{<,>}` (`!=`); per the dossier widening rule it is WIDENED to the
              universal `{<,=,>}` to keep path-consistency complete, and every
              widening is COUNTED (bite lost by convex restriction).
  * ALLEN  -- Allen interval algebra (13 base relations). Composition is generated
              by the ENDPOINT method (enumerate weak orders of the six interval
              endpoints) and unit-tested against the dossier's GQR cells. Full Allen
              PC is sound but INCOMPLETE -> closure-detectable inconsistency is a
              LOWER BOUND there.

Closure operators:
  * close_triangle      -- the per-triangle length-2 path narrowing used for the
                           frontier metrics (path = compose(AB,BC); inter = path & AC).
  * pc2_full            -- Mackworth PC-2 worklist closure to fixpoint (OUR METHOD).
  * naive_single_pass   -- BASELINE: one pass of length-2 path intersections at the
                           query edge, no fixpoint / no re-propagation
                           ("Path-of-Thoughts + one intersection step").
"""

import itertools
from collections import deque
from typing import Iterable

# ----------------------------------------------------------------------------------
# Allen-13 base relations (qualreas-compatible symbols) and their endpoint geometry.
# Interval X = (Xs, Xe) with Xs < Xe. relation(X, Y) defined on (Xs, Xe, Ys, Ye).
# ----------------------------------------------------------------------------------
ALLEN_BASE = ["B", "BI", "D", "DI", "O", "OI", "M", "MI", "S", "SI", "F", "FI", "E"]
ALLEN_CONVERSE = {"B": "BI", "BI": "B", "D": "DI", "DI": "D", "O": "OI", "OI": "O",
                  "M": "MI", "MI": "M", "S": "SI", "SI": "S", "F": "FI", "FI": "F", "E": "E"}


def _allen_rel(xs: int, xe: int, ys: int, ye: int) -> str | None:
    """Atomic Allen relation of interval X=(xs,xe) to Y=(ys,ye) from endpoint ranks.

    Assumes proper intervals (xs<xe, ys<ye). Returns one of the 13 base symbols.
    """
    if not (xs < xe and ys < ye):
        return None
    if xs == ys and xe == ye:
        return "E"
    if xe < ys:
        return "B"
    if ye < xs:
        return "BI"
    if xe == ys:
        return "M"
    if ye == xs:
        return "MI"
    if xs == ys:
        return "S" if xe < ye else "SI"
    if xe == ye:
        return "F" if xs > ys else "FI"
    if xs < ys and ye < xe:
        return "DI"
    if ys < xs and xe < ye:
        return "D"
    if xs < ys < xe < ye:
        return "O"
    if ys < xs < ye < xe:
        return "OI"
    return None  # unreachable for proper intervals


def _build_allen_compose() -> dict[tuple[str, str], frozenset]:
    """Generate the Allen base x base composition table via endpoint enumeration.

    Enumerate every rank assignment of the 6 endpoints (As,Ae,Bs,Be,Cs,Ce) in 0..5
    (ties allowed; only relative order matters). For each that yields proper intervals,
    record comp[r(A,B)][r(B,C)] |= {r(A,C)}.
    """
    comp: dict[tuple[str, str], set] = {(a, b): set() for a in ALLEN_BASE for b in ALLEN_BASE}
    for asg in itertools.product(range(6), repeat=6):
        As, Ae, Bs, Be, Cs, Ce = asg
        if not (As < Ae and Bs < Be and Cs < Ce):
            continue
        rab = _allen_rel(As, Ae, Bs, Be)
        rbc = _allen_rel(Bs, Be, Cs, Ce)
        rac = _allen_rel(As, Ae, Cs, Ce)
        if rab is None or rbc is None or rac is None:
            continue
        comp[(rab, rbc)].add(rac)
    return {k: frozenset(v) for k, v in comp.items()}


# ----------------------------------------------------------------------------------
# Point algebra over start-points.
# ----------------------------------------------------------------------------------
POINT_BASE = ["<", "=", ">"]
POINT_CONVERSE = {"<": ">", "=": "=", ">": "<"}
# GQR point.comp (verified): = is identity; <o<={<}; >o>={>}; <o>=>o<=universal.
POINT_COMPOSE = {
    ("=", "="): frozenset({"="}),
    ("<", "="): frozenset({"<"}), ("=", "<"): frozenset({"<"}),
    (">", "="): frozenset({">"}), ("=", ">"): frozenset({">"}),
    ("<", "<"): frozenset({"<"}),
    (">", ">"): frozenset({">"}),
    ("<", ">"): frozenset({"<", "=", ">"}),
    (">", "<"): frozenset({"<", "=", ">"}),
}
POINT_NONCONVEX = frozenset({"<", ">"})  # the only non-convex point relation (`!=`)


class Algebra:
    """A qualitative calculus with relation sets stored as frozensets of base symbols."""

    def __init__(self, name: str, base: list[str], converse: dict[str, str],
                 compose_bb: dict[tuple[str, str], frozenset], identity: frozenset,
                 convex_widen: frozenset | None = None):
        self.name = name
        self.base = list(base)
        self.universe = frozenset(base)
        self.empty = frozenset()
        self.identity = frozenset(identity)
        self._conv = dict(converse)
        self._comp = dict(compose_bb)
        # `convex_widen` (point algebra only): the unique non-convex relation that must
        # be widened to the universal set to keep PC complete. None => no widening.
        self._nonconvex = convex_widen

    # ---- set ops -----------------------------------------------------------------
    def converse(self, s: frozenset) -> frozenset:
        return frozenset(self._conv[r] for r in s)

    def compose(self, a: frozenset, b: frozenset) -> frozenset:
        if not a or not b:
            return self.empty
        out: set = set()
        for x in a:
            for y in b:
                out |= self._comp[(x, y)]
        return frozenset(out)

    def is_nonconvex(self, s: frozenset) -> bool:
        return self._nonconvex is not None and s == self._nonconvex

    def widen(self, s: frozenset) -> tuple[frozenset, bool]:
        """Return (possibly-widened set, fired?). Widening only happens for the
        unique non-convex relation of a convex algebra (point: `{<,>}` -> universe)."""
        if self._nonconvex is not None and s == self._nonconvex:
            return self.universe, True
        return s, False

    def label(self, s: frozenset) -> str:
        if not s:
            return "EMPTY"
        if s == self.universe:
            return "UNIVERSE"
        return "|".join(r for r in self.base if r in s)


def build_point_algebra() -> Algebra:
    return Algebra("POINT", POINT_BASE, POINT_CONVERSE, POINT_COMPOSE,
                   frozenset({"="}), convex_widen=POINT_NONCONVEX)


def build_allen_algebra() -> Algebra:
    return Algebra("ALLEN", ALLEN_BASE, ALLEN_CONVERSE, _build_allen_compose(),
                   frozenset({"E"}), convex_widen=None)


# ----------------------------------------------------------------------------------
# Triangle closure (frontier metric primitive)
# ----------------------------------------------------------------------------------
def close_triangle(alg: Algebra, ab: frozenset, bc: frozenset, ac: frozenset):
    """Length-2 path A-B-C narrowing the query edge A-C.

    path  = compose(ab, bc)                    (with convex widening applied + counted)
    inter = path & ac                          (widening applied + counted)
    Returns dict: path, inter, empty(collapse), singleton, n_widen.
    """
    n_widen = 0
    path = alg.compose(ab, bc)
    path, w = alg.widen(path)
    n_widen += int(w)
    inter = path & ac
    inter, w = alg.widen(inter)
    n_widen += int(w)
    return {
        "path": path,
        "inter": inter,
        "empty": len(inter) == 0,
        "singleton": len(inter) == 1,
        "n_widen": n_widen,
    }


# ----------------------------------------------------------------------------------
# Full QCN + closure variants (reused by whole-document consistency checks)
# ----------------------------------------------------------------------------------
class QCN:
    """Constraint network: dense node list, edges = relation-set frozensets.

    Missing edge => universe. Invariants on set_edge: M[j][i]==converse(M[i][j]),
    M[i][i]==identity.
    """

    def __init__(self, alg: Algebra, nodes: list):
        self.alg = alg
        self.nodes = list(nodes)
        self.n = len(self.nodes)
        self.index = {nd: i for i, nd in enumerate(self.nodes)}
        U = alg.universe
        self.M = [[U] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.M[i][i] = alg.identity
        self.nbrs: list[set] = [set() for _ in range(self.n)]

    def set_edge(self, i: int, j: int, s: frozenset) -> None:
        if i == j:
            return
        self.M[i][j] = s
        self.M[j][i] = self.alg.converse(s)
        if s != self.alg.universe:
            self.nbrs[i].add(j); self.nbrs[j].add(i)
        else:
            self.nbrs[i].discard(j); self.nbrs[j].discard(i)

    def get(self, i: int, j: int) -> frozenset:
        return self.M[i][j]

    def known_edges(self) -> list[tuple[int, int]]:
        U = self.alg.universe
        return [(i, j) for i in range(self.n) for j in range(i + 1, self.n) if self.M[i][j] != U]


def pc2_full(qcn: QCN):
    """OUR METHOD: Mackworth PC-2 worklist closure to fixpoint.

    Returns (consistent: bool, n_fired). Empty edge => inconsistent (Mode-B certificate).
    Convex widening (point algebra) applied to every refined edge and absorbed silently
    (it can only enlarge a set, never empties it).
    """
    alg = qcn.alg
    U = alg.universe
    M = qcn.M
    nbrs = qcn.nbrs
    Q = deque()
    inq = set()
    for (i, j) in qcn.known_edges():
        Q.append((i, j)); inq.add((i, j))
        Q.append((j, i)); inq.add((j, i))
    n_fired = 0

    def enqueue(a, b):
        if (a, b) not in inq:
            inq.add((a, b)); Q.append((a, b))

    while Q:
        i, j = Q.popleft(); inq.discard((i, j))
        rij = M[i][j]
        if rij == U:
            continue
        for k in list(nbrs[j]):
            if k == i:
                continue
            comp = alg.compose(rij, M[j][k])
            new = M[i][k] & comp
            new, _ = alg.widen(new)
            if new != M[i][k]:
                if not new:
                    return False, n_fired
                M[i][k] = new; M[k][i] = alg.converse(new)
                nbrs[i].add(k); nbrs[k].add(i)
                n_fired += 1
                enqueue(i, k); enqueue(k, i)
        for k in list(nbrs[i]):
            if k == j:
                continue
            comp = alg.compose(M[k][i], rij)
            new = M[k][j] & comp
            new, _ = alg.widen(new)
            if new != M[k][j]:
                if not new:
                    return False, n_fired
                M[k][j] = new; M[j][k] = alg.converse(new)
                nbrs[k].add(j); nbrs[j].add(k)
                n_fired += 1
                enqueue(k, j); enqueue(j, k)
    return True, n_fired


def naive_single_pass(qcn: QCN, u: int, v: int) -> frozenset:
    """BASELINE: single pass of length-2 path compositions at the query edge (u,v).

    Intersects compose(R(u,w), R(w,v)) over intermediate w with informative u-w & w-v.
    NO fixpoint, NO re-propagation.
    """
    alg = qcn.alg
    U = alg.universe
    M = qcn.M
    R = U
    for w in qcn.nbrs[u]:
        if w in (u, v):
            continue
        if M[w][v] != U:
            R = R & alg.compose(M[u][w], M[w][v])
            R, _ = alg.widen(R)
            if not R:
                return alg.empty
    return R

## 2 · Reproduce the closure **emit / abstain** decisions on real LLM reads

Each stored trace-graph holds the *real* span-local LLM reads (`induced_path_edges`) of the
events on a query's reasoning path. We build a QCN from those reads, run PC-2 to fixpoint, and
read the **held-out** query edge — mirroring `temporal_core.run_query`'s Mode-A. The live engine
must reproduce the published `modeA` decision (and the number of PC-2 refinements that fired).

In [ ]:
# POINT <-> coarse label maps (from temporal_core.py) + a thin Mode-A wrapper that
# mirrors temporal_core.run_query: build the QCN from the span-local reads, close, read query.
PT = build_point_algebra()
POINT_SET_TO_COARSE = {
    frozenset({"<"}): "before", frozenset({">"}): "after",
    frozenset({"="}): "simultaneous", frozenset({"<", "="}): "includes",
    frozenset({"=", ">"}): "is_included",
}

def point_to_coarse(R):
    return POINT_SET_TO_COARSE.get(frozenset(R))

def _label_to_pointset(lbl):
    return frozenset({"<", "=", ">"}) if lbl == "UNIVERSE" else frozenset(lbl.split("|"))

def closure_decision(trace):
    edges = trace["induced_path_edges"]
    qx, qy = trace["query"]
    nodes = sorted({e["from"] for e in edges} | {e["to"] for e in edges} | {qx, qy})
    qcn = QCN(PT, nodes); idx = qcn.index
    for e in edges:
        if {e["from"], e["to"]} == {qx, qy}:        # NEVER seed the held-out query edge
            continue
        qcn.set_edge(idx[e["from"]], idx[e["to"]], _label_to_pointset(e["local_read_point"]))
    naive_set = naive_single_pass(qcn, idx[qx], idx[qy])   # baseline (one pass, no fixpoint)
    consistent, n_fired = pc2_full(qcn)                    # OUR METHOD (PC-2 to fixpoint)
    if not consistent:
        return {"modeA": "COLLAPSE(Mode-B)", "n_fired": n_fired,
                "naive": point_to_coarse(naive_set) or "ABSTAIN"}
    R = qcn.get(idx[qx], idx[qy])
    coarse = point_to_coarse(R)
    modeA = coarse if (len(R) == 1 and coarse) else "ABSTAIN"   # singleton -> emit, else abstain
    return {"modeA": modeA, "n_fired": n_fired,
            "naive": point_to_coarse(naive_set) or "ABSTAIN"}


print(f"{'doc':<10}{'kind':<11}{'query (held-out)':<18}{'gold':<14}"
      f"{'naive':<14}{'live modeA':<14}{'published':<14}{'pc2':<5}match")
print("-" * 98)
n_ok = 0
for tg in data["temporal_trace_graphs"][:MAX_TRACE_GRAPHS]:
    dec = closure_decision(tg)
    qx, qy = (n.split("::")[-1] for n in tg["query"])
    match = (dec["modeA"] == tg["modeA_closure_pred"]) and (dec["n_fired"] == tg["n_pc2_fired"])
    n_ok += int(match)
    print(f"{tg['docid'][:8]:<10}{tg['kind'][12:]:<11}{qx + '->' + qy:<18}"
          f"{str(tg['gold_coarse']):<14}{dec['naive']:<14}{dec['modeA']:<14}"
          f"{tg['modeA_closure_pred']:<14}{dec['n_fired']:<5}{'OK' if match else 'X'}")
print("-" * 98)
print(f"live engine reproduced {n_ok}/{min(MAX_TRACE_GRAPHS, len(data['temporal_trace_graphs']))} "
      f"published Mode-A decisions exactly")

### Inspect one trace: the span-local reads that drive a decision

For an **emit** case every read on the path is a singleton `<` (`before`), so composing
`A < M` with `M < B` narrows the held-out `A → B` edge to `{<}`. For an **abstain** case some
reads are `UNIVERSE` (the text under-determines the order), so closure cannot reach a
singleton and Mode-A faithfully abstains instead of guessing.

In [ ]:
def show_trace(tg):
    qx, qy = (n.split("::")[-1] for n in tg["query"])
    print(f"[{tg['docid']}]  query (held out): {qx} -> {qy}   gold={tg['gold_coarse']}   "
          f"published outcome={tg['modeA_outcome']}")
    print("  span-local LLM reads on the induced path:")
    for e in tg["induced_path_edges"]:
        a, b = e["from"].split("::")[-1], e["to"].split("::")[-1]
        print(f"    {a:>8} -> {b:<8}  point_read={e['local_read_point']:<10}"
              f"coarse={str(e['local_read_coarse']):<12} (conf {e['conf']})")

emit_tg = next(t for t in data["temporal_trace_graphs"] if t["kind"] == "trace_graph_narrowing")
abst_tg = next(t for t in data["temporal_trace_graphs"] if t["kind"] == "trace_graph_abstain")
print("=== EMIT (closure narrows to a singleton) ===")
show_trace(emit_tg)
print("\n=== ABSTAIN (under-determined -> faithful abstention) ===")
show_trace(abst_tg)

## 3 · The headline: **hallucination (confident-wrong) reduction**

A *confident-wrong* answer is one a method **commits** to (does not abstain) that disagrees
with gold. We recompute it over the cached per-query predictions for every method, using the
artifact's own `cw_coverage` / `reduction_block` (verbatim). The reduction is the raw LLM's
confident-wrong rate **minus** Mode-A's, with each method's **coverage** reported beside it —
the certified pipeline abstains where the reads under-determine, so it is wrong far less often.

In [ ]:
# verbatim from method.py -------------------------------------------------------
def cw_coverage(pairs):
    """pairs: list of (answered: bool, wrong: bool). Confident-wrong AS a risk-coverage
    operating point: coverage = fraction of the POOL the method names; confident-wrong rate
    is reported BOTH over the pool (comparable across methods) AND among answered."""
    n = len(pairs)
    n_ans = sum(1 for a, w in pairs if a)
    n_wrong = sum(1 for a, w in pairs if a and w)
    return {
        "n_queries": n,
        "coverage": round(n_ans / n, 4) if n else 0.0,
        "abstention": round(1 - n_ans / n, 4) if n else 0.0,
        "n_answered": n_ans,
        "confident_wrong_count": n_wrong,
        "confident_wrong_rate_over_pool": round(n_wrong / n, 4) if n else 0.0,
        "confident_wrong_rate_among_answered": (round(n_wrong / n_ans, 4) if n_ans else None),
    }

def reduction_block(modeA_cw, raw_cw, label):
    """Hallucination (confident-wrong) reduction WITH coverage beside every number."""
    red = modeA_cw["confident_wrong_rate_over_pool"]
    red = raw_cw["confident_wrong_rate_over_pool"] - red
    return {
        "comparison": f"Mode-A (closure-certified) vs {label}",
        "confident_wrong_rate_modeA_over_pool": modeA_cw["confident_wrong_rate_over_pool"],
        "confident_wrong_rate_raw_over_pool": raw_cw["confident_wrong_rate_over_pool"],
        "hallucination_reduction_over_pool": round(red, 4),
        "coverage_modeA": modeA_cw["coverage"],
        "coverage_raw": raw_cw["coverage"],
        "n_queries": modeA_cw["n_queries"],
    }

# A prediction is "answered" unless it is the ABSTAIN sentinel; "wrong" if it disagrees w/ gold.
def to_pair(pred, gold):
    answered = pred not in (None, "ABSTAIN", "None")
    return (answered, answered and pred != gold)

rows = data["examples"][:MAX_TEMPORAL_EXAMPLES]
by_doc = defaultdict(list)
for e in rows:
    by_doc[e["metadata_docid"]].append(e)

print(f"{'document':<26}{'modeA cw':<11}{'raw cw':<10}{'reduction':<11}"
      f"{'cov modeA':<11}{'cov raw':<9}")
print("-" * 78)
doc_reduction = {}
for doc, es in by_doc.items():
    cwA = cw_coverage([to_pair(e["predict_modeA"],      e["output"]) for e in es])
    cwF = cw_coverage([to_pair(e["predict_raw_fulldoc"], e["output"]) for e in es])
    rb = reduction_block(cwA, cwF, "whole-document raw LLM")
    doc_reduction[doc] = rb
    print(f"{doc:<26}{cwA['confident_wrong_rate_over_pool']:<11}"
          f"{cwF['confident_wrong_rate_over_pool']:<10}"
          f"{rb['hallucination_reduction_over_pool']:<11}"
          f"{rb['coverage_modeA']:<11}{rb['coverage_raw']:<9}")

# pooled over all replayed rows
allA = cw_coverage([to_pair(e["predict_modeA"],       e["output"]) for e in rows])
allF = cw_coverage([to_pair(e["predict_raw_fulldoc"], e["output"]) for e in rows])
allL = cw_coverage([to_pair(e["predict_raw_local"],   e["output"]) for e in rows])
pooled = reduction_block(allA, allF, "whole-document raw LLM")
print("-" * 78)
print(f"{'POOLED':<26}{allA['confident_wrong_rate_over_pool']:<11}"
      f"{allF['confident_wrong_rate_over_pool']:<10}"
      f"{pooled['hallucination_reduction_over_pool']:<11}"
      f"{pooled['coverage_modeA']:<11}{pooled['coverage_raw']:<9}")
print(f"\nMode-A confident-wrong: {allA['confident_wrong_count']}/{allA['n_queries']} "
      f"| whole-doc raw: {allF['confident_wrong_count']}/{allF['n_queries']} "
      f"| span-local raw: {allL['confident_wrong_count']}/{allL['n_queries']}")

## 4 · The kinship forward-closure engine (`kinship.py`, verbatim)

The temporal arm shows faithfulness-by-abstention on *real* text where extraction is the
ceiling. The kinship arm uses a **memorized finite composition table**, so the symbolic
pipeline works *fully* and we can watch the certificate end-to-end. The sound closure for a
finite table is a **forward least-fixpoint UNION derivation** (not PC-2, which is unsound here).
This is the artifact's `kinship.py`, unchanged.

In [ ]:
#!/usr/bin/env python3
"""Kinship finite-composition closure engine (the symbolic half of the pipeline).

CLUTRR's kinship calculus is a FINITE COMPOSITION TABLE over 11 abstract relation
TYPES -- and, as the dataset card states verbatim, it is NOT a full relation algebra:
"no general intersection/converse closure beyond these rules". A naive port of the
temporal PC-2 engine (Mackworth converse-intersection path-consistency) is therefore
UNSOUND here -- closing the converse constraints makes ~13% of GOLD-clean chains
spuriously collapse to EMPTY, because two derivation orders can yield two different
(individually valid) relations that the table does not reconcile. (We verified this
directly: PC-2-style closure gives 100% accuracy WHEN it answers but only 0.87
singleton-rate, with 2153/16131 clean rows collapsing.)

The SOUND closure for a finite composition table is a FORWARD LEAST-FIXPOINT
*UNION* DERIVATION over DEFINED compositions only -- exactly the backward/forward
chaining that produces CLUTRR's own `gold_proof`, and exactly what the emitted Prolog
`derive/3` predicate computes:

  D[(a,b)] = { t : t derivable for directed pair a->b by composing atomic links }

  * seed: every atomic edge a->b:t adds t to D[(a,b)] and conv(t) to D[(b,a)];
  * close: while changing, for every a-b-c, for t1 in D[(a,b)], t2 in D[(b,c)],
    if rules[t1][t2]=t3 is DEFINED, add t3 to D[(a,c)] (and conv(t3) to D[(c,a)]);
    UNDEFINED compositions add nothing (SOUND: "unknown", never a wrong fact).

OUTPUT CONTRACT (disjunction-preserving Mode-A / abstain-on-collapse Mode-B):
  * |D[query]| == 1  -> EMIT the relation (covered).            [unique derivation]
  * |D[query]| >  1  -> ABSTAIN (Mode-B CONFLICT certificate).  [incompatible derivations]
  * |D[query]| == 0  -> ABSTAIN (no connecting path = universe). [absent-relation / underdetermined]

This contract is hallucination-safe by construction: with no connecting path (the
absent-relation case, entities in different components) D is empty, so the engine
NEVER invents a kinship -- it abstains. The naive single-pass baseline uses ONLY the
seed (atomic) edges and one composition step, so it resolves hop-2 chains but abstains
on hop>=3 (the iteration contrast), while the full fixpoint derives the intermediate
composite edges first and resolves the whole chain.
"""

from collections import defaultdict, deque
from typing import Iterable


class Kinship:
    """Finite kinship composition calculus parsed from the dataset composition table."""

    def __init__(self, comp_table: dict):
        rt = comp_table["relation_types"]
        self.base: list[str] = list(rt.keys())  # 11 abstract relation types
        self.universe = frozenset(self.base)
        self.empty = frozenset()
        self.symmetric_types = set(comp_table["symmetric_types"])  # {'sibling','SO'}
        self.inv: dict[str, str] = {}
        for a, b in comp_table["inverse_pairs"].items():
            self.inv[a] = b
            self.inv[b] = a
        self.composition_rules = comp_table["composition_rules"]
        self.surface_forms = comp_table["surface_forms"]
        self.surface_reverse = comp_table["surface_reverse"]
        self.label_map = comp_table.get("label_map", {})
        self.label_map_reverse = comp_table.get("label_map_reverse", {})
        # ---- total converse over every base type (sound; no empties) ----
        self._conv: dict[str, str] = {}
        for t in self.base:
            if t in self.symmetric_types:
                self._conv[t] = t
            elif t in self.inv:
                self._conv[t] = self.inv[t]
            elif t == "sibling-in-law":
                # brother/sister-in-law are mutual: converse(sibling-in-law)=sibling-in-law.
                self._conv[t] = t
            else:
                self._conv[t] = t  # sound self-converse fallback (never reached for the 11 types)

    # ------------------------------------------------------------------ ops
    def conv_type(self, t: str) -> str:
        return self._conv[t]

    def compose_types(self, t1: str, t2: str):
        """Defined composition rules[t1][t2]=t3, else None (UNDEFINED == 'unknown')."""
        return self.composition_rules.get(t1, {}).get(t2)

    def label(self, s) -> str:
        s = frozenset(s)
        if not s:
            return "EMPTY"
        if s == self.universe:
            return "UNIVERSE"
        return "|".join(t for t in self.base if t in s)

    # ------------------------------------------------------------- surface words
    def surface(self, rel_type: str, gender: str) -> str:
        g = "male" if str(gender).lower().startswith("m") else "female"
        sf = self.surface_forms.get(rel_type)
        if not sf:
            return rel_type
        return sf.get(g, sf.get("male", rel_type))

    def surface_to_type(self, surface_word: str):
        """Return (relation_type, implied_gender) or None for an unknown word."""
        w = str(surface_word).strip().lower()
        rev = self.surface_reverse.get(w)
        if rev is None:
            return None
        return rev[0], rev[1]


# --------------------------------------------------------------------------- #
# Forward least-fixpoint UNION derivation (the sound closure for the finite table)
# --------------------------------------------------------------------------- #
def _seed(kin: Kinship, atomic_edges: list[dict]):
    """Seed D with atomic edges + their converses. Returns (D, nbrs).
    D[(a,b)] = set of types; nbrs[a] = set of directed successors."""
    D: dict = defaultdict(set)
    nbrs: dict = defaultdict(set)

    def add(a, b, t):
        if t not in D[(a, b)]:
            D[(a, b)].add(t)
            nbrs[a].add(b)

    for e in atomic_edges:
        t = e["type"]
        if t not in kin.base:
            continue
        a, b = e["a"], e["b"]
        if a == b:
            continue
        add(a, b, t)
        add(b, a, kin.conv_type(t))
    return D, nbrs


def forward_closure(kin: Kinship, atomic_edges: list[dict], with_prov: bool = False):
    """Forward least-fixpoint union derivation. Returns (D, nbrs, n_fired) or, with
    with_prov, (D, nbrs, n_fired, prov) where prov[(a,c,t3)] = (a,b,c,t1,t2,t3) records
    the FIRST composition that produced type t3 on pair (a,c) (a directed-edge of the
    proof DAG; seed edges map to None).

    D[(a,b)] holds EVERY relation type derivable for the directed pair a->b; closed
    under defined composition + converse. n_fired = number of new type-additions."""
    D, nbrs = _seed(kin, atomic_edges)
    prov: dict = {}
    if with_prov:
        for (a, b), ts in D.items():
            for t in ts:
                prov.setdefault((a, b, t), None)
    Q = deque(D.keys())
    inq = set(D.keys())
    n_fired = 0

    def push(p):
        if p not in inq:
            inq.add(p)
            Q.append(p)

    def emit(a, c, t3, provtuple):
        nonlocal n_fired
        grew = False
        if t3 not in D[(a, c)]:
            D[(a, c)].add(t3)
            nbrs[a].add(c)
            if with_prov:
                prov.setdefault((a, c, t3), provtuple)
            n_fired += 1
            grew = True
        ct3 = kin.conv_type(t3)
        if ct3 not in D[(c, a)]:
            D[(c, a)].add(ct3)
            nbrs[c].add(a)
            if with_prov:
                prov.setdefault((c, a, ct3), (c, a, a, ct3, None, ct3))  # converse marker
        if grew:
            push((a, c)); push((c, a))

    while Q:
        (a, b) = Q.popleft()
        inq.discard((a, b))
        tab = list(D[(a, b)])
        # extend a->b with b->c  =>  a->c
        for c in list(nbrs[b]):
            if c == a:
                continue
            for t1 in tab:
                for t2 in list(D[(b, c)]):
                    t3 = kin.compose_types(t1, t2)
                    if t3 is not None:
                        emit(a, c, t3, (a, b, c, t1, t2, t3))
        # extend z->a with a->b  =>  z->b   (a is the middle)
        for z in list(nbrs[a]):
            if z == b:
                continue
            for t1 in list(D[(z, a)]):
                for t2 in tab:
                    t3 = kin.compose_types(t1, t2)
                    if t3 is not None:
                        emit(z, b, t3, (z, a, b, t1, t2, t3))
    if with_prov:
        return D, nbrs, n_fired, prov
    return D, nbrs, n_fired


def naive_single_pass(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt) -> set:
    """BASELINE: ONE composition pass at the query edge using ONLY seed (atomic) edges.

    R = union over intermediates w of {rules[t1][t2] : t1 in seed(u,w), t2 in seed(w,v)}.
    NO fixpoint, NO derived edges. On a hop-k chain only the hop-2 case has an
    intermediate w with BOTH atomic links to the endpoints, so naive resolves hop-2 but
    derives nothing (-> abstain) on hop>=3."""
    D, nbrs = _seed(kin, atomic_edges)
    R: set = set()
    for w in nbrs.get(qsrc, ()):
        if w in (qsrc, qtgt):
            continue
        if (w, qtgt) in D:
            for t1 in D[(qsrc, w)]:
                for t2 in D[(w, qtgt)]:
                    t3 = kin.compose_types(t1, t2)
                    if t3 is not None:
                        R.add(t3)
    return R


# --------------------------------------------------------------------------- #
# Query wrappers with the Mode-A / Mode-B output contract
# --------------------------------------------------------------------------- #
def _answer_from_set(kin: Kinship, R: set) -> dict:
    R = set(R)
    n = len(R)
    if n == 1:
        t = next(iter(R))
        return {"types": sorted(R), "singleton": True, "answer_type": t,
                "n_derivations": n, "mode_b_conflict": False, "no_path": False}
    if n == 0:
        return {"types": [], "singleton": False, "answer_type": None,
                "n_derivations": 0, "mode_b_conflict": False, "no_path": True}
    # n > 1 : incompatible derivations => Mode-B conflict
    rep = sorted(R, key=lambda t: kin.base.index(t))[0]  # deterministic representative
    return {"types": sorted(R), "singleton": False, "answer_type": rep,
            "n_derivations": n, "mode_b_conflict": True, "no_path": False}


def query_modeA(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt) -> dict:
    """Mode-A forward-closure query. Returns the output-contract decision + n_fired."""
    D, nbrs, n_fired = forward_closure(kin, atomic_edges)
    R = D.get((qsrc, qtgt), set())
    out = _answer_from_set(kin, R)
    out["n_fired"] = n_fired
    return out


def query_naive(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt) -> dict:
    """Naive single-pass query (fresh seed only)."""
    R = naive_single_pass(kin, atomic_edges, qsrc, qtgt)
    return _answer_from_set(kin, R)


def simple_paths_names(atomic_edges: list[dict], qsrc, qtgt, max_paths: int = 3,
                       max_len: int = 12):
    """Up to `max_paths` simple undirected entity paths qsrc..qtgt over the atomic-edge
    graph (feeds Path-of-Thoughts). Returns lists of node names, shortest first."""
    adj: dict = {}
    for e in atomic_edges:
        adj.setdefault(e["a"], set()).add(e["b"])
        adj.setdefault(e["b"], set()).add(e["a"])
    if qsrc not in adj or qtgt not in adj:
        return []
    paths: list[list] = []
    stack = [(qsrc, [qsrc])]
    while stack and len(paths) < max_paths * 4:
        node, path = stack.pop()
        if len(path) > max_len:
            continue
        for nb in sorted(adj.get(node, ())):
            if nb == qtgt:
                paths.append(path + [nb])
            elif nb not in path:
                stack.append((nb, path + [nb]))
    paths.sort(key=len)
    # de-dup
    seen = set(); uniq = []
    for p in paths:
        k = tuple(p)
        if k not in seen:
            seen.add(k); uniq.append(p)
    return uniq[:max_paths]


def derivation_trace(kin: Kinship, atomic_edges: list[dict], qsrc, qtgt,
                     max_steps: int = 60):
    """Reconstruct ONE concrete derivation for (qsrc->qtgt) for the trace-graph:
    which (t1 o t2 -> t3) compositions fire, mirroring the gold backward proof.
    Returns a list of {a,b,c,t1,t2,t3} steps producing the answer type, or [] if the
    query is not a unique-derivation singleton."""
    D, nbrs, _, prov = forward_closure(kin, atomic_edges, with_prov=True)
    target = D.get((qsrc, qtgt), set())
    if len(target) != 1:
        return []
    goal_type = next(iter(target))
    steps = []
    stack = [(qsrc, qtgt, goal_type)]
    seen = set()
    while stack and len(steps) < max_steps:
        key = stack.pop()
        if key in seen:
            continue
        seen.add(key)
        p = prov.get(key)
        if p is None:
            continue  # seed edge (atomic fact) -- a leaf of the proof DAG
        a, b, c, t1, t2, t3 = p
        if t2 is None:
            # converse marker: unfold to the forward edge (b->a : conv(t3))
            stack.append((c, a, kin.conv_type(t3)))
            continue
        steps.append({"a": a, "b": b, "c": c, "t1": t1, "t2": t2, "t3": t3})
        stack.append((a, b, t1))
        stack.append((b, c, t2))
    steps.reverse()
    return steps

### Run worked kinship chains: multi-hop **emit**, **iteration contrast**, **absent-abstain**

* a **3-hop** chain the full closure resolves but the naive single pass cannot (iteration matters),
* a **2-hop** chain both resolve, and
* an **absent** pair in disjoint sub-stories → empty derivation set → **abstain by construction**
  (the structural hallucination guarantee).

In [ ]:
kin = Kinship(data["kinship_composition_table"])
SURFACE = {"un": "niece/nephew", "inv-un": "aunt/uncle", "grand": "grandchild",
           "inv-grand": "grandparent", "child": "child", "inv-child": "parent",
           "sibling": "sibling", "SO": "spouse"}

for ex in data["kinship_worked_examples"][:MAX_KINSHIP_EXAMPLES]:
    src, tgt = ex["query"]
    mA = query_modeA(kin, ex["atomic_edges"], src, tgt)
    nv = query_naive(kin, ex["atomic_edges"], src, tgt)
    trace = derivation_trace(kin, ex["atomic_edges"], src, tgt)
    outcome = ("EMIT " + mA["answer_type"] if mA["singleton"]
               else ("ABSTAIN (Mode-B conflict)" if mA["mode_b_conflict"]
                     else "ABSTAIN (no path)"))
    print(f"### {ex['name']}   {src} -> {tgt}   (gold: {ex['gold_surface']})")
    print(f"    atomic edges: " +
          ", ".join(f"{e['a']}-{e['type']}->{e['b']}" for e in ex["atomic_edges"]))
    print(f"    full closure (Mode-A): {outcome}"
          + (f"  [= {SURFACE.get(mA['answer_type'], mA['answer_type'])}]" if mA["singleton"] else ""))
    print(f"    naive single-pass    : "
          + ("EMIT " + nv["answer_type"] if nv["singleton"] else "ABSTAIN")
          + ("   <-- iteration contrast: closure resolves what one pass cannot"
             if mA["singleton"] and not nv["singleton"] else ""))
    if trace:
        print("    derivation (composition steps that fire):")
        for s in trace:
            print(f"        {s['a']}--{s['t1']}-->{s['b']}  o  {s['b']}--{s['t2']}-->{s['c']}"
                  f"   =>   {s['a']}--{s['t3']}-->{s['c']}")
    print()

## 5 · Discharge the certificate as a **runnable SWI-Prolog** program (`prolog_kinship.py`)

The closed network is emitted as a forward-union least-fixpoint Prolog program whose
`setof(R, der(src,R,tgt), Rs)` read-back is identical to the engine's derivation set. It is
executed in `swipl` when installed and **cross-checked against the engine** (`matches_engine`);
otherwise the Python engine is authoritative. This is `prolog_kinship.py`, unchanged
(`_atom` inlined).

In [ ]:
#!/usr/bin/env python3
"""Forward-closure (least-fixpoint UNION) kinship discharge in ACTUAL SWI-Prolog.

The iter-3 prolog.py emits a simple-path right-fold `solve_/4`, which is INCOMPLETE for
the finite kinship composition table: a single association order can hit an UNDEFINED
composition and miss a derivation the engine's full fixpoint finds (this is exactly the
iteration / closure advantage). For a COHERENT human-auditable trace that reproduces the
certified engine, we emit a program that computes the SAME forward-union least fixpoint:

  der(A,T,B)  closed under  der(A,T1,B), der(B,T2,C), comp(T1,T2,T3) => der(A,T3,C)
  and under converse (every edge asserted both directions),

then reads back  setof(R, der(qsrc,qtgt,R), Rs)  --  identical to engine.query_modeA's D set.
The discharge runs swipl as a subprocess (honest python-checked fallback if unavailable),
and is cross-checked against the engine's forward_closure type set (matches_engine).
"""

import shutil
import subprocess
import tempfile
from pathlib import Path

def _atom(x) -> str:
    """Quote any token as a Prolog atom (Capitalized names + hyphenated types must
    be quoted). Escape embedded quotes/backslashes."""
    s = str(x).replace('\\', '\\\\').replace("'", "\\'")
    return f"'{s}'"



def emit_fixpoint_program(kin, edges, qsrc, qtgt) -> str:
    lines = [
        "% Closure-certified kinship deduction -- forward-union least-fixpoint trace program.",
        "% der(A,T,B): T derivable for the directed pair A->B; closed under the finite",
        "% composition table comp/3 + total converse conv/2 (both-direction seeding).",
        ":- dynamic der/3.",
        "% ---- composition rules comp(T1,T2,T3) ----",
    ]
    for t1, row in kin.composition_rules.items():
        for t2, t3 in row.items():
            lines.append(f"comp({_atom(t1)},{_atom(t2)},{_atom(t3)}).")
    lines.append("% ---- total converse conv(T,Tc) ----")
    for t, tc in kin._conv.items():
        lines.append(f"conv({_atom(t)},{_atom(tc)}).")
    lines.append("% ---- seed: extracted atomic edges seed(A,T,B) ----")
    seen = set()
    for e in edges:
        t = e["type"]
        a, b = e["a"], e["b"]
        if t not in kin.base or a == b:
            continue
        if (a, t, b) in seen:
            continue
        seen.add((a, t, b))
        lines.append(f"seed({_atom(a)},{_atom(t)},{_atom(b)}).")
    lines += [
        "% ---- add a directed edge + its converse (idempotent) ----",
        "add_edge(A,T,B) :- ( der(A,T,B) -> true ; assertz(der(A,T,B)) ),",
        "                   conv(T,Tc), ( der(B,Tc,A) -> true ; assertz(der(B,Tc,A)) ).",
        "init_seed :- forall(seed(A,T,B), add_edge(A,T,B)).",
        "% ---- batch forward-chaining to least fixpoint ----",
        "one_pass :- findall(d(A,T3,C),",
        "             ( der(A,T1,B), der(B,T2,C), comp(T1,T2,T3), \\+ der(A,T3,C) ), L0),",
        "           sort(L0, L), ( L == [] -> true",
        "             ; ( forall(member(d(A,T,C),L), add_edge(A,T,C)), one_pass ) ).",
        f"run :- init_seed, one_pass,",
        f"       ( setof(R, der({_atom(qsrc)},R,{_atom(qtgt)}), Rs) -> true ; Rs = [] ),",
        "       forall(member(R,Rs), (write('RESULT:'), write(R), nl)), halt.",
        ":- initialization(run).",
    ]
    return "\n".join(lines) + "\n"


def _parse_results(stdout: str):
    out = set()
    for ln in (stdout or "").splitlines():
        ln = ln.strip()
        if ln.startswith("RESULT:"):
            out.add(ln[len("RESULT:"):].strip())
    return sorted(out)


def discharge_fixpoint(kin, edges, qsrc, qtgt, engine_types, outpath: Path = None,
                       timeout: float = 25.0) -> dict:
    """Run the forward-closure program in swipl; cross-check vs the engine's type set."""
    program = emit_fixpoint_program(kin, edges, qsrc, qtgt)
    engine_types = sorted(set(engine_types))
    if outpath is not None:
        Path(outpath).write_text(program)
    swipl = shutil.which("swipl")
    if not swipl:
        return {"engine": "python-fallback", "executed_in_swipl": False,
                "note": "swipl unavailable; engine forward_closure is authoritative",
                "prolog_results": None, "engine_types": engine_types,
                "matches_engine": None, "program_chars": len(program),
                "program": program}
    tmp = None
    path = str(outpath) if outpath is not None else None
    if path is None:
        with tempfile.NamedTemporaryFile("w", suffix=".pl", delete=False) as fh:
            fh.write(program); path = fh.name; tmp = path
    try:
        proc = subprocess.run([swipl, "-q", "-g", "true", "-t", "halt", "-s", path],
                              capture_output=True, text=True, timeout=timeout)
        results = _parse_results(proc.stdout)
        executed = (proc.returncode == 0)
        return {"engine": "swipl-subprocess", "executed_in_swipl": executed,
                "prolog_results": results, "engine_types": engine_types,
                "matches_engine": (results == engine_types) if executed else None,
                "stdout_tail": (proc.stdout or "")[-400:], "stderr_tail": (proc.stderr or "")[-300:],
                "exit_code": proc.returncode, "program_chars": len(program), "program": program}
    except subprocess.TimeoutExpired:
        return {"engine": "swipl-subprocess", "executed_in_swipl": False, "prolog_results": None,
                "engine_types": engine_types, "matches_engine": None, "note": "timeout",
                "program_chars": len(program), "program": program}
    finally:
        if tmp:
            try:
                Path(tmp).unlink()
            except OSError:
                pass

In [ ]:
# Emit + discharge the program for the multi-hop chain (and the absent pair to show abstain).
import shutil

def discharge(ex):
    src, tgt = ex["query"]
    engine = query_modeA(kin, ex["atomic_edges"], src, tgt)
    res = discharge_fixpoint(kin, ex["atomic_edges"], src, tgt, engine["types"])
    return engine, res

ex = data["kinship_worked_examples"][0]            # the 3-hop niece chain
engine, res = discharge(ex)
print(f"chain: {ex['query'][0]} -> {ex['query'][1]}   engine derivation set: {engine['types']}")
print(f"swipl available: {shutil.which('swipl') is not None}  "
      f"| executed_in_swipl: {res.get('executed_in_swipl')}  "
      f"| prolog_results: {res.get('prolog_results')}  "
      f"| matches_engine: {res.get('matches_engine')}")
print("\n----- emitted SWI-Prolog program (excerpt) -----")
prog = res["program"]
print("\n".join(prog.splitlines()[:22]))
print("...")
print("\n".join(prog.splitlines()[-6:]))

# absent pair: empty derivation set in BOTH engine and Prolog -> abstain by construction
absent = next(e for e in data["kinship_worked_examples"] if e["name"] == "absent_disjoint")
aeng, ares = discharge(absent)
print(f"\nabsent pair {absent['query'][0]} -> {absent['query'][1]}: "
      f"engine types={aeng['types']}  prolog_results={ares.get('prolog_results')}  "
      f"-> {'ABSTAIN (0 confident-wrong by construction)' if not aeng['types'] else 'EMIT'}")

## 6 · Summary & visualization

Left: per-document **confident-wrong rate** — whole-document raw LLM (always commits) vs the
closure-certified Mode-A (abstains where under-determined) — and the resulting **hallucination
reduction**. Right: the operational case-study ranges reported in the paper. Every Mode-A
number carries its coverage, because faithfulness here is bought with abstention.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# --- left: per-doc confident-wrong rate (raw vs modeA) ---
docs = list(doc_reduction.keys())
short = [d.split(".")[0][:10] for d in docs]
raw_cw = [doc_reduction[d]["confident_wrong_rate_raw_over_pool"] for d in docs]
mA_cw = [doc_reduction[d]["confident_wrong_rate_modeA_over_pool"] for d in docs]
red = [doc_reduction[d]["hallucination_reduction_over_pool"] for d in docs]
x = np.arange(len(docs)); w = 0.38
ax = axes[0]
ax.bar(x - w / 2, raw_cw, w, label="raw LLM (whole-doc)", color="#d1495b")
ax.bar(x + w / 2, mA_cw, w, label="Mode-A (closure-certified)", color="#2a9d8f")
for i, r in enumerate(red):
    ax.annotate(f"-{r:.2f}", (i, max(raw_cw[i], mA_cw[i]) + 0.02), ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(short, rotation=15)
ax.set_ylabel("confident-wrong rate (over pool)")
ax.set_title("Hallucination reduction per document\n(label = reduction)")
ax.legend(fontsize=9); ax.set_ylim(0, max(raw_cw) + 0.12)

# --- right: case-study summary ranges ---
ax = axes[1]
cs = data["case_study_summary"]["temporal"]
metrics = ["atomic recall", "most-likely prec.", "halluc. reduction\n(vs whole-doc raw)"]
keys = ["atomic_disjunctive_recall", "atomic_most_likely_precision",
        "hallucination_reduction_vs_fulldoc_raw"]
means = [cs[k]["mean"] for k in keys]
los = [cs[k]["mean"] - cs[k]["min"] for k in keys]
his = [cs[k]["max"] - cs[k]["mean"] for k in keys]
y = np.arange(len(metrics))
ax.barh(y, means, xerr=[los, his], color="#457b9d", capsize=5, height=0.5)
for i, m in enumerate(means):
    ax.text(m + 0.02, i, f"{m:.2f}", va="center", fontsize=9)
ax.set_yticks(y); ax.set_yticklabels(metrics)
ax.set_xlim(0, 1.05); ax.invert_yaxis()
ax.set_title(f"Temporal case study (5 docs, 150 queries)\nmean with min/max range")
ax.set_xlabel("value")
plt.tight_layout(); plt.show()

# --- text summary ---
ks = data["case_study_summary"]["kinship"]
print("KINSHIP arm (3 concatenated ~3000-char docs):")
print(f"  atomic recall (extraction bottleneck): mean {ks['atomic_recall']['mean']:.2f}")
print(f"  within-story multi-hop selective accuracy: {ks['within_story_selective_accuracy']}")
print(f"  absent-relation pairs abstained: {ks['absent_pairs_abstained']}/{ks['absent_pairs_total']}"
      f"  -> confident-wrong on absent pairs: {ks['absent_confident_wrong_total']}")
print("\nTakeaway: the closure certificate ABSTAINS where evidence is insufficient, cutting")
print("confident-wrong (hallucination) rate vs a raw LLM that always commits.")